# Phase 3 / D1 — Crash-Risk-Premium-Adjusted Carry (option-implied skew battery)

Stages 1–6 showed the 2007–2026 carry premium is an EM phenomenon and that every *standard*
overlay — crash hedges (St3), optimization (St4), momentum (St5), regime timing (St6) — fails to
beat the simple vol-targeted inverse-vol carry book net of costs. **D1** asks whether the FX
**option surface** — the market's explicit pricing of currency crash risk (25Δ risk reversals,
standardised by ATM) — turns into carry alpha, or is another honest null.

**The bar (falsifiable).** Beat *both* the simple vol-targeted book (ALL net Sharpe **0.466**) *and*
the per-currency-RR-hedged book (**0.457**), net of costs, with Newey–West significance — or report
a null. Guardrails §6: no lookahead (signals sampled month-end, `ffill().shift(1)`, trailing windows
only), gross AND net (`forward_halfspreads`+`roundtrip_cost`), IR vs FXCTEM8, window 2007-05→2026-06.

**The battery** — matched **21-name** option universe, quintile inverse-vol, vol-targeted 10%:

| variant | signal | competing thesis |
|---|---|---|
| (a) `iskew` | RR/ATM smile skew (crash-positive) | Farhi–Gabaix: high RR ⇒ paid for risk ⇒ **long**; Brunnermeier: high RR ⇒ warning ⇒ avoid |
| (b) `blendhi`/`blendlo` | `0.5·z(carry) ± 0.5·z(skew)` | carry tilted **toward** vs **away from** priced crash |
| (c) `clean` | carry ⟂ implied skew (`xs_residual`) | Jurek "crash-neutral" clean carry |
| (d) `srp` | `z(realised skew) + z(implied skew)` | Li–Sarno–Zinna skewness risk premium — flagship + spanning |

The RR is sign-normalised **crash-positive**, i.e. the *negative* of risk-neutral skewness, so LSZ's
`SRP = physical − risk-neutral` becomes `z(realised) + z(implied)`. A long/short book is antisymmetric
under sort reversal, so for (a)/(d) "both directions" = the reported sign vs its negation; for (b) the
two tilts are genuinely different books, both run. Comparator for NW alpha = the **matched** vanilla
carry; the published 0.466 / 0.457 are the absolute bars.

In [1]:
# ===== 0. Setup =====
from pathlib import Path
import numpy as np, pandas as pd
import fx_utils as fx

OUT = Path('outputs')
g10_px = fx.load_wide('g10_fx_spot_forward')
em_px  = fx.load_wide('em_fx_spot_forward')
spots  = fx.spots_usd_per_fx(g10_px, em_px)
carry  = fx.carry_panel(g10_px, em_px, tenor='1M')
xret   = fx.excess_returns(spots, carry)
bmk    = fx.benchmark_returns()
hs_out, hs_pts = fx.forward_halfspreads('1M')

BENCH      = 'FXCTEM8'
VOL_TARGET = 0.10
N_BUCKETS  = 5
SKEW_WIN   = 252
WINDOW     = slice('2007-05-01', '2026-06-30')

# Matched universe: tradable 27 names ∩ option (RR) coverage = 21 names.
UNIVERSE_ALL = [c for c in xret.columns if c not in ('HKD', 'DKK', 'CNY')]
RR_COVERED   = fx.vol_surface_panel('RR', '1M').columns
U21          = [c for c in UNIVERSE_ALL if c in RR_COVERED]
print('xret', xret.shape, '| ALL', len(UNIVERSE_ALL), '| U21', len(U21), '| bench', BENCH)
print('U21:', U21)

xret (5094, 29) | ALL 27 | U21 21 | bench FXCTEM8
U21: ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'JPY', 'NOK', 'NZD', 'SEK', 'BRL', 'CNH', 'HUF', 'ILS', 'INR', 'KRW', 'MXN', 'PLN', 'SGD', 'THB', 'TRY', 'ZAR']


## 1. Signals and weight panels

Three new pure helpers drive the battery: `implied_skew_panel` (RR/ATM, crash-positive),
`realized_skew_panel` (trailing physical skew of `xret`), `xs_residual` (cross-sectional clean
carry). Each signal is substituted for `carry_ann` in `carry_portfolio` (the library's sort contract
— there is no callable hook), then vol-targeted to 10%.

In [2]:
# ===== 1. Signals and weight panels =====
# Option-implied crash skew (RR/ATM) aligned to the return calendar so blends/SRP combine cleanly.
iskew  = fx.implied_skew_panel(standardize=True).reindex(xret.index)[U21]
rskew  = fx.realized_skew_panel(xret, window=SKEW_WIN)[U21]
carryU = carry[U21]

zc, zi, zr = fx.zscore_xs(carryU), fx.zscore_xs(iskew), fx.zscore_xs(rskew)
signals = {
    'carry':   carryU,                        # matched vanilla carry (baseline / NW comparator)
    'iskew':   iskew,                         # (a) standalone implied skew, long high RR (Farhi)
    'blendhi': 0.5 * zc + 0.5 * zi,           # (b) carry tilted toward high priced-crash
    'blendlo': 0.5 * zc - 0.5 * zi,           # (b) carry tilted away (Jurek/Brunnermeier)
    'clean':   fx.xs_residual(carryU, iskew), # (c) clean carry = carry ⟂ implied skew
    'srp':     zr + zi,                       # (d) SRP = z(physical) − z(RN); RN = −iskew ⇒ +z(iskew)
}

def vt(w):
    return fx.vol_target_weights(w, xret, target=VOL_TARGET)

w = {k: vt(fx.carry_portfolio(sig, xret, n_buckets=N_BUCKETS, universe=U21, weighting='inv_vol'))
     for k, sig in signals.items()}
# ALL-27 vanilla carry: reconciliation anchor to Stage 3/4 (net Sharpe ~0.466).
w['all_carry'] = vt(fx.carry_portfolio(carry, xret, n_buckets=N_BUCKETS,
                                       universe=UNIVERSE_ALL, weighting='inv_vol'))
# SRP robustness on a 126d realised-skew window (reported, NOT tuned on).
w['srp126'] = vt(fx.carry_portfolio(fx.zscore_xs(fx.realized_skew_panel(xret, 126)[U21]) + zi,
                                    xret, n_buckets=N_BUCKETS, universe=U21, weighting='inv_vol'))
print('weight panels:', list(w))

weight panels: ['carry', 'iskew', 'blendhi', 'blendlo', 'clean', 'srp', 'all_carry', 'srp126']


## 2. Tracks — gross and net of costs

`net = gross − roundtrip_cost` (new notional pays the outright half-spread; maintained notional rolls
at the points half-spread). Column convention `{universe}_{variant}_{basis}`.

In [3]:
# ===== 2. Gross / net tracks =====
LABEL = {'carry': 'U21_carry', 'iskew': 'U21_iskew', 'blendhi': 'U21_blendhi',
         'blendlo': 'U21_blendlo', 'clean': 'U21_clean', 'srp': 'U21_srp',
         'srp126': 'U21_srp126', 'all_carry': 'ALL_carry'}
tracks, costs = {}, {}
for k, wk in w.items():
    lbl = LABEL[k]
    g = fx.portfolio_returns(wk, xret).rename(f'{lbl}_gross')
    c = fx.roundtrip_cost(wk, hs_out, hs_pts)
    tracks[g.name] = g
    tracks[f'{lbl}_net'] = (g - c).rename(f'{lbl}_net')
    costs[lbl] = c
daily = pd.concat(list(tracks.values()) + [bmk[BENCH]], axis=1).loc[WINDOW]
print('daily panel', daily.shape, '| tracks', len(tracks))

daily panel (5008, 17) | tracks 16


## 3. Statistics — full metrics, IR, turnover, cost drag, NW alpha vs matched carry

IR vs FXCTEM8. NW (5-lag HAC) alpha of every net track on the **matched** `U21_carry_net`.

In [4]:
# ===== 3. Statistics =====
strat_cols = [c for c in daily.columns if c != BENCH]
summary = pd.concat([
    fx.summary_stats(daily[strat_cols], benchmark=bmk[BENCH]),
    fx.summary_stats(daily[[BENCH]]),
])

base = daily['U21_carry_net']                       # matched vanilla carry = NW comparator
meta = {}
for k, lbl in LABEL.items():
    to = fx.turnover(w[k].loc[WINDOW])
    for basis in ('gross', 'net'):
        col = f'{lbl}_{basis}'
        row = {'avg_monthly_turnover': to}
        if basis == 'net':
            n = daily[col]
            c_win = costs[lbl].reindex(n.index)
            row['ann_cost_drag'] = float(c_win[n.notna()].mean() * fx.ANN_DAYS)
            if col != 'U21_carry_net':
                reg = fx.nw_regression(n, base.to_frame('carry'), lags=5)
                row['alpha_vs_carry_ann'] = reg['alpha_ann'] if reg else np.nan
                row['t_alpha_vs_carry']   = reg['alpha_t'] if reg else np.nan
        meta[col] = row
summary = summary.join(pd.DataFrame(meta).T)

show = ['sharpe', 'info_ratio', 'max_drawdown', 'CVaR_99', 'skew',
        'avg_monthly_turnover', 'ann_cost_drag', 't_alpha_vs_carry']
summary[show].round(3)

,sharpe,info_ratio,max_drawdown,CVaR_99,skew,avg_monthly_turnover,ann_cost_drag,t_alpha_vs_carry
U21_carry_gross,0.619438,0.507146,-0.235066,0.030108,-0.733681,0.468,NaN,NaN
U21_carry_net,0.496238,0.381011,-0.257568,0.030108,-0.734441,0.468,0.014,NaN
U21_iskew_gross,0.322936,0.199221,-0.330854,0.034403,-1.072748,1.049,NaN,NaN
U21_iskew_net,0.131613,0.014075,-0.512905,0.034514,-1.068003,1.049,0.023,-1.586
U21_blendhi_gross,0.28897,0.171758,-0.376199,0.032804,-0.908562,0.728,NaN,NaN
U21_blendhi_net,0.147417,0.027281,-0.473417,0.032862,-0.904548,0.728,0.016,-2.840
U21_blendlo_gross,0.064412,-0.046875,-0.382851,0.026894,-0.060672,1.449,NaN,NaN
U21_blendlo_net,-0.212661,-0.267326,-0.557454,0.026935,-0.049461,1.449,0.032,-1.204
U21_clean_gross,0.205439,0.071756,-0.332261,0.028609,-0.488631,1.171,NaN,NaN
U21_clean_net,-0.030866,-0.13055,-0.423818,0.028747,-0.481872,1.171,0.027,-1.196


## 4. SRP-vs-carry spanning — the headline novelty test

Li–Sarno–Zinna (2023) *claim* SRP subsumes carry (carry's alpha vanishes, SRP's survives). We test
it **both ways** on U21 unit gross-2/net-0 long/short factor books (gross returns, the spanning
convention), NW 5 lags. This specific claim was single-source/unverified — **test, do not assert.**

In [5]:
# ===== 4. SRP vs carry spanning =====
def factor(sig, name):
    wf = fx.carry_portfolio(sig, xret, n_buckets=N_BUCKETS, universe=U21, weighting='inv_vol')
    return fx.portfolio_returns(wf, xret, name).loc[WINDOW]

carry_f = factor(signals['carry'], 'CARRY')
srp_f   = factor(signals['srp'], 'SRP')

def span_row(y, X, xname):
    r = fx.nw_regression(y, X, lags=5)
    return {'alpha_ann': r['alpha_ann'], 'alpha_t': r['alpha_t'],
            'beta': r[f'beta_{xname}'], 't_beta': r[f't_{xname}'], 'r2': r['r2'], 'n': r['n']}

span = pd.DataFrame({
    'SRP~CARRY': span_row(srp_f, carry_f.to_frame('CARRY'), 'CARRY'),   # does SRP survive carry?
    'CARRY~SRP': span_row(carry_f, srp_f.to_frame('SRP'), 'SRP'),       # does carry survive SRP?
}).T[['alpha_ann', 'alpha_t', 'beta', 't_beta', 'r2', 'n']]
print('factor gross Sharpe: carry',
      round(float(fx.summary_stats(carry_f.to_frame('c')).loc['c', 'sharpe']), 3),
      '| srp', round(float(fx.summary_stats(srp_f.to_frame('s')).loc['s', 'sharpe']), 3),
      '| corr', round(carry_f.corr(srp_f), 3))
span.round(4)

factor gross Sharpe: carry 0.487 | srp 0.111 | corr 0.404


,alpha_ann,alpha_t,beta,t_beta,r2,n
SRP~CARRY,-0.0048,-0.3811,0.2862,10.1572,0.1633,4956.0
CARRY~SRP,0.0376,2.1904,0.5708,13.1926,0.1633,4956.0


## 5. Validation — universe, no-lookahead, reconciliation, window

In [6]:
# ===== 5. Validation =====
# (i) matched universe is exactly the 21 option-covered tradable names
assert sorted(U21) == sorted(['AUD','CAD','CHF','EUR','GBP','JPY','NOK','NZD','SEK',
    'BRL','CNH','HUF','ILS','INR','KRW','MXN','PLN','SGD','THB','TRY','ZAR']), 'U21 mismatch'

# (ii) no-lookahead: realised skew on data truncated at `cut` matches the full panel on overlap
cut = '2015-06-30'
rs_tr = fx.realized_skew_panel(xret.loc[:cut], SKEW_WIN)[U21].dropna(how='all')
assert np.allclose(rs_tr.iloc[-20:].values,
                   rskew.reindex(rs_tr.index).iloc[-20:].values, equal_nan=True), 'lookahead in realized_skew_panel'

# (iii) no-lookahead: SRP weights on truncated data match the full run on an inner window
srp_full = fx.carry_portfolio(signals['srp'], xret, n_buckets=N_BUCKETS, universe=U21, weighting='inv_vol')
srp_tr_sig = (fx.zscore_xs(fx.realized_skew_panel(xret.loc[:cut], SKEW_WIN)[U21])
              + fx.zscore_xs(fx.implied_skew_panel(standardize=True).reindex(xret.loc[:cut].index)[U21]))
srp_tr = fx.carry_portfolio(srp_tr_sig, xret.loc[:cut], n_buckets=N_BUCKETS, universe=U21, weighting='inv_vol')
chk = slice('2010-01-01', '2015-05-31')
assert np.allclose(srp_full.loc[chk].fillna(0).values,
                   srp_tr.loc[chk].reindex(columns=srp_full.columns).fillna(0).values, atol=1e-10), 'lookahead in srp weights'

# (iv) reconcile ALL-27 inv_vol-net Sharpe to the committed Stage-4 number (~0.466)
s4 = pd.read_csv(OUT / 'stage4_weighting_comparison.csv', index_col=0)
a = float(summary.loc['ALL_carry_net', 'sharpe']); b = float(s4.loc['ALL_inv_vol_net', 'sharpe'])
print(f'ALL-27 inv_vol-net Sharpe {a:.4f} vs Stage-4 {b:.4f} (d={a - b:+.5f})')
assert abs(a - b) < 1e-3, 'ALL-27 carry path drifted from Stage 4'

# (v) window end, and a loose start bound (SRP books warm up ~4 months on the 252d skew)
strat = [i for i in summary.index if i.startswith(('U21_', 'ALL_'))]
assert all(str(summary.loc[i, 'end']) == '2026-06-30' for i in strat), 'window end'
assert all(str(summary.loc[i, 'start']) <= '2007-10-31' for i in strat), 'window start'
non_base = [i for i in strat if i.endswith('_net') and i != 'U21_carry_net']
assert summary.loc[non_base, 't_alpha_vs_carry'].notna().all(), 'missing NW alpha'
print('checks passed')

ALL-27 inv_vol-net Sharpe 0.4659 vs Stage-4 0.4659 (d=+0.00000)
checks passed


## 6. Output + acceptance checks

In [7]:
# ===== 6. Output =====
OUT.mkdir(exist_ok=True)
summary.to_csv(OUT / 'skew_carry_comparison.csv')
span.to_csv(OUT / 'srp_carry_spanning.csv')
net_cols = [c for c in daily.columns if c.endswith('_net')]
corr = daily[net_cols].corr()
corr.to_csv(OUT / 'skew_track_correlation.csv')
assert (OUT / 'skew_carry_comparison.csv').exists() and (OUT / 'srp_carry_spanning.csv').exists()
print('wrote: skew_carry_comparison.csv, srp_carry_spanning.csv, skew_track_correlation.csv')

wrote: skew_carry_comparison.csv, srp_carry_spanning.csv, skew_track_correlation.csv


## 7. Verdict — does option-implied crash-risk pricing beat vanilla carry net of costs?

**REJECT — another honest null.** On the matched 21-name option universe the vanilla vol-targeted
carry book earns net Sharpe **0.496** — itself *above* both published bars (0.466, 0.457): dropping
the six thin, optionless EM (CLP/COP/IDR/MYR/PEN/PHP) if anything *helps*. Every option-implied-skew
variant lands far below it, none with positive Newey–West alpha:

| variant | net Sharpe | α vs carry (ann) | t | verdict |
|---|---|---|---|---|
| (a) implied skew, long high RR | **0.13** | −2.8% | −1.6 | reject |
| (b) carry tilted **toward** crash (`blendhi`) | **0.15** | −3.3% | −2.8 | reject (sig. worse) |
| (b) carry tilted **away** (`blendlo`) | **−0.21** | −3.2% | −1.2 | reject |
| (c) clean carry (Jurek residual) | **−0.03** | −2.9% | −1.2 | reject |
| (d) SRP (Li–Sarno–Zinna) | **−0.09** | −3.3% | −1.5 | reject |

- **Contested RR direction — settled weakly for Farhi–Gabaix, but it doesn't matter.** Long-high-RR
  is the positive sign (+0.13); its negation (short high RR) is −0.13. Either way it is a fraction of
  carry with no alpha. The Brunnermeier "avoid expensive insurance" tilt (`blendlo`) is the worst book.
- **Clean carry collapses**, exactly as Jurek warns: a dollar-neutral book (ours is net-0 by
  construction) stripped of the crash-priced component keeps ~none of the premium.
- **Flagship SRP fails both directions** (−0.09 / −0.40 reversed) and, crucially, **the Li–Sarno–Zinna
  spanning claim does not replicate — it reverses.** On 2007–2026, U21: SRP earns *zero* alpha over
  carry (**α −0.5%/yr, t −0.4**) while **carry keeps a significant alpha over SRP (α +3.8%/yr, t +2.2)**.
  Here **carry subsumes SRP**, the opposite of LSZ's single-source claim — the headline novelty
  hypothesis is falsified on this sample. (Robustness: the 126d-skew SRP is likewise negative, so this
  is not a window artefact.)

**Bottom line.** The option surface's explicit crash-risk pricing (real, per Stage 2) does **not**
convert into carry alpha net of costs. D1 is a clean null — a valid deliverable. It also *sharpens*
the project's through-line: crash risk explains part of *why* carry is compensated, but is not itself
a tradable edge over the simple book.

In [8]:
# ===== 7. Verdict block — net decision metrics + the two bars =====
show = ['sharpe', 'info_ratio', 'max_drawdown', 'CVaR_99', 'skew', 'ann_cost_drag', 't_alpha_vs_carry']
rows = ['U21_carry_net', 'U21_iskew_net', 'U21_blendhi_net', 'U21_blendlo_net',
        'U21_clean_net', 'U21_srp_net', 'U21_srp126_net', 'ALL_carry_net']
print('=== D1 skew battery — NET (matched 21-name universe) ===')
print(summary.loc[rows, show].round(3))
print(f"\nBars to beat: simple vol-targeted ALL 0.466 | per-ccy RR 0.457 "
      f"| matched U21 carry {float(summary.loc['U21_carry_net', 'sharpe']):.3f}")
print('\n=== SRP vs carry spanning (gross factor returns, NW 5 lags) ===')
print(span.round(4))
print('\nVerdict: REJECT — no skew variant beats the bars; SRP does NOT subsume carry (carry subsumes SRP).')

=== D1 skew battery — NET (matched 21-name universe) ===
                   sharpe info_ratio max_drawdown   CVaR_99      skew  \
U21_carry_net    0.496238   0.381011    -0.257568  0.030108 -0.734441   
U21_iskew_net    0.131613   0.014075    -0.512905  0.034514 -1.068003   
U21_blendhi_net  0.147417   0.027281    -0.473417  0.032862 -0.904548   
U21_blendlo_net -0.212661  -0.267326    -0.557454  0.026935 -0.049461   
U21_clean_net   -0.030866   -0.13055    -0.423818  0.028747 -0.481872   
U21_srp_net     -0.090586  -0.183346    -0.490212  0.026944 -0.482982   
U21_srp126_net  -0.030767  -0.124834    -0.562963  0.027354  -0.09561   
ALL_carry_net    0.465923   0.337642    -0.293185  0.029199 -0.648005   

                 ann_cost_drag  t_alpha_vs_carry  
U21_carry_net            0.014               NaN  
U21_iskew_net            0.023            -1.586  
U21_blendhi_net          0.016            -2.840  
U21_blendlo_net          0.032            -1.204  
U21_clean_net            0.027